# 17 - Counterfactual Charging Explanations

This notebook turns optimised charging into simple recommendations.

## Purpose

Compare immediate charging with a feasible optimised schedule. The result is a practical suggestion for when charging could move.

In [1]:
from pathlib import Path
import json
import math
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)


def find_project_root(start: Path | None = None) -> Path:
    """Find the repository root from either the root or Notebooks directory."""
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "Data").exists() and (candidate / "Notebooks").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing Data/ and Notebooks/.")


ROOT = find_project_root()
DATA_DIR = ROOT / "Data"
RESULTS_DIR = DATA_DIR / "Results"
PROCESSED_DIR = DATA_DIR / "Processed"
FIGURES_DIR = ROOT / "Figures"
MODELS_DIR = ROOT / "Models" / "Forecasting"

for directory in [RESULTS_DIR, PROCESSED_DIR, FIGURES_DIR, MODELS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

INTERVAL_MINUTES = 5
INTERVAL_HOURS = INTERVAL_MINUTES / 60
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

print(f"Project root: {ROOT}")

Project root: /Users/mac/Send-Ev-Project/SEND-EV-Project


In [2]:
def save_table(df: pd.DataFrame, path: Path) -> Path:
    """Save Parquet where available, otherwise save CSV with the same stem."""
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.suffix.lower() == ".parquet":
        try:
            df.to_parquet(path, index=False)
            print(f"Saved: {path.relative_to(ROOT)}")
            return path
        except (ImportError, ModuleNotFoundError) as exc:
            csv_path = path.with_suffix(".csv")
            df.to_csv(csv_path, index=False)
            print(f"Parquet engine unavailable ({exc.__class__.__name__}); saved CSV: {csv_path.relative_to(ROOT)}")
            return csv_path
    df.to_csv(path, index=False)
    print(f"Saved: {path.relative_to(ROOT)}")
    return path


def read_table(path: Path, parse_dates: list[str] | None = None) -> pd.DataFrame:
    """Read a Parquet/CSV output regardless of which format was written."""
    candidates = [path]
    if path.suffix.lower() == ".parquet":
        candidates.append(path.with_suffix(".csv"))
    elif path.suffix.lower() == ".csv":
        candidates.append(path.with_suffix(".parquet"))

    parquet_error: Exception | None = None

    for candidate in candidates:
        if candidate.exists():
            if candidate.suffix.lower() == ".parquet":
                try:
                    return pd.read_parquet(candidate)
                except (ImportError, ModuleNotFoundError) as exc:
                    parquet_error = exc
                    continue
            return pd.read_csv(candidate, parse_dates=parse_dates)

    if parquet_error is not None:
        raise ImportError(
            "A Parquet file exists, but pandas cannot read it and no CSV fallback was found. "
            "Install pyarrow or fastparquet, or rerun the previous notebook to create the CSV fallback."
        ) from parquet_error

    raise FileNotFoundError(f"None of these files exists: {candidates}")

In [3]:
scenarios = read_table(DATA_DIR / "ACN" / "processed" / "keele_ev_scenarios.parquet")
baseline = read_table(RESULTS_DIR / "baseline_vehicle_schedule.parquet")
optimised = read_table(RESULTS_DIR / "optimised_vehicle_schedule.parquet")
send = read_table(PROCESSED_DIR / "send_solcast_surplus_5min.parquet", parse_dates=["DateTime"])

for column in ["arrival_time", "departure_time", "keele_date"]:
    scenarios[column] = pd.to_datetime(scenarios[column], errors="coerce")
for frame in [baseline, optimised, send]:
    frame["DateTime"] = pd.to_datetime(frame["DateTime"], errors="coerce")

baseline = baseline[baseline["strategy"].eq("immediate")].copy()
print("Optimised strategies:", optimised["strategy"].drop_duplicates().tolist())

Optimised strategies: ['optimised_actual']


## 1. Explanation helpers

In [4]:
def clock_text(value) -> str:
    if pd.isna(value):
        return "no charging"
    return pd.Timestamp(value).strftime("%H:%M")


def schedule_vector(frame: pd.DataFrame, grid: pd.DatetimeIndex) -> pd.Series:
    if frame.empty:
        return pd.Series(0.0, index=grid)
    return frame.groupby("DateTime")["charging_kw"].sum().reindex(grid, fill_value=0.0)


def validate_recommended_schedule(
    vehicle: pd.Series,
    schedule: pd.DataFrame,
) -> dict:
    if schedule.empty:
        return {
            "connection_feasible": True,
            "power_feasible": True,
            "energy_feasible": False,
            "delivered_kwh": 0.0,
        }
    connection = schedule["DateTime"].ge(vehicle["arrival_time"]) & schedule["DateTime"].lt(vehicle["departure_time"])
    power = schedule["charging_kw"].le(float(vehicle["max_power_kw"]) + 1e-5)
    delivered = schedule["charging_kw"].sum() * INTERVAL_HOURS * float(vehicle["efficiency"])
    return {
        "connection_feasible": bool(connection.all()),
        "power_feasible": bool(power.all()),
        "energy_feasible": bool(delivered >= float(vehicle["energy_required_kwh"]) * 0.99),
        "delivered_kwh": float(delivered),
    }

## 2. Generate vehicle-level counterfactuals

In [5]:
surplus_lookup = send.set_index("DateTime")["surplus_kw"]
common_scenarios = sorted(set(baseline["scenario_id"]) & set(optimised["scenario_id"]))
explanation_rows = []

for scenario_id in common_scenarios:
    scenario = scenarios[scenarios["scenario_id"].eq(scenario_id)].copy()
    base_scenario = baseline[baseline["scenario_id"].eq(scenario_id)]

    for strategy, recommended_scenario in optimised[optimised["scenario_id"].eq(scenario_id)].groupby("strategy"):
        for _, vehicle in scenario.iterrows():
            vehicle_id = vehicle["vehicle_id"]
            original = base_scenario[base_scenario["vehicle_id"].eq(vehicle_id)].copy()
            recommended = recommended_scenario[recommended_scenario["vehicle_id"].eq(vehicle_id)].copy()

            grid_start = pd.Timestamp(vehicle["arrival_time"]).floor(f"{INTERVAL_MINUTES}min")
            grid_end = pd.Timestamp(vehicle["departure_time"]).ceil(f"{INTERVAL_MINUTES}min")
            grid = pd.date_range(grid_start, grid_end, freq=f"{INTERVAL_MINUTES}min", inclusive="left")
            original_vector = schedule_vector(original, grid)
            recommended_vector = schedule_vector(recommended, grid)
            surplus = surplus_lookup.reindex(grid).interpolate(limit=3).fillna(0)

            original_start = original.loc[original["charging_kw"].gt(1e-5), "DateTime"].min()
            recommended_start = recommended.loc[recommended["charging_kw"].gt(1e-5), "DateTime"].min()
            original_end = original.loc[original["charging_kw"].gt(1e-5), "DateTime"].max()
            recommended_end = recommended.loc[recommended["charging_kw"].gt(1e-5), "DateTime"].max()

            original_surplus_window_energy = (
                original_vector[surplus.gt(0)].sum() * INTERVAL_HOURS
            )
            recommended_surplus_window_energy = (
                recommended_vector[surplus.gt(0)].sum() * INTERVAL_HOURS
            )
            additional_surplus_window_energy = (
                recommended_surplus_window_energy - original_surplus_window_energy
            )
            changed_intervals = int((original_vector - recommended_vector).abs().gt(0.05).sum())
            schedule_distance_kwh = float((original_vector - recommended_vector).abs().sum() * INTERVAL_HOURS)
            validation = validate_recommended_schedule(vehicle, recommended)

            if pd.isna(recommended_start):
                explanation = (
                    f"No feasible charging was allocated to {vehicle_id}; "
                    f"{vehicle['energy_required_kwh']:.1f} kWh was requested before {clock_text(vehicle['departure_time'])}."
                )
            else:
                delay_minutes = (
                    (recommended_start - original_start).total_seconds() / 60
                    if pd.notna(original_start) else np.nan
                )
                average_power = recommended["charging_kw"].mean() if not recommended.empty else 0.0
                explanation = (
                    f"For {vehicle_id}, move the charging start from {clock_text(original_start)} to "
                    f"{clock_text(recommended_start)} and use an average of {average_power:.1f} kW until approximately "
                    f"{clock_text(recommended_end + pd.Timedelta(minutes=INTERVAL_MINUTES) if pd.notna(recommended_end) else recommended_end)}. "
                    f"This shifts {additional_surplus_window_energy:.1f} kWh more charging into observed surplus intervals "
                    f"while targeting completion before the {clock_text(vehicle['departure_time'])} departure."
                )

            explanation_rows.append({
                "scenario_id": scenario_id,
                "strategy": strategy,
                "vehicle_id": vehicle_id,
                "arrival_time": vehicle["arrival_time"],
                "departure_time": vehicle["departure_time"],
                "energy_required_kwh": vehicle["energy_required_kwh"],
                "original_start": original_start,
                "recommended_start": recommended_start,
                "start_delay_minutes": delay_minutes if pd.notna(recommended_start) and pd.notna(original_start) else np.nan,
                "changed_intervals": changed_intervals,
                "schedule_distance_kwh": schedule_distance_kwh,
                "additional_surplus_window_energy_kwh": additional_surplus_window_energy,
                **validation,
                "counterfactual_feasible": all([
                    validation["connection_feasible"],
                    validation["power_feasible"],
                    validation["energy_feasible"],
                ]),
                "explanation": explanation,
            })

counterfactuals = pd.DataFrame(explanation_rows)
print(f"Generated {len(counterfactuals):,} vehicle-level explanations")
display(
    counterfactuals.sort_values("additional_surplus_window_energy_kwh", ascending=False)[
        ["scenario_id", "strategy", "vehicle_id", "counterfactual_feasible", "explanation"]
    ].head(10)
)

Generated 424 vehicle-level explanations


,scenario_id,strategy,vehicle_id,counterfactual_feasible,explanation
83,20230707_pooled_F25_P050_S43,optimised_actual,EV010,True,"For EV010, move the charging start from 07:00 ..."
195,20230707_pooled_F50_P025_S43,optimised_actual,EV010,True,"For EV010, move the charging start from 07:00 ..."
114,20230707_pooled_F25_P075_S43,optimised_actual,EV010,True,"For EV010, move the charging start from 07:00 ..."
49,20230707_pooled_F10_P100_S43,optimised_actual,EV010,True,"For EV010, move the charging start from 07:00 ..."
232,20230707_pooled_F50_P050_S43,optimised_actual,EV010,True,"For EV010, move the charging start from 07:00 ..."
295,20230707_pooled_F50_P075_S43,optimised_actual,EV010,True,"For EV010, move the charging start from 07:00 ..."
28,20230707_pooled_F10_P075_S43,optimised_actual,EV007,True,"For EV007, move the charging start from 06:45 ..."
80,20230707_pooled_F25_P050_S43,optimised_actual,EV007,True,"For EV007, move the charging start from 06:45 ..."
192,20230707_pooled_F50_P025_S43,optimised_actual,EV007,True,"For EV007, move the charging start from 06:45 ..."
111,20230707_pooled_F25_P075_S43,optimised_actual,EV007,True,"For EV007, move the charging start from 06:45 ..."


## 3. Evaluate explanation feasibility and sparsity

In [6]:
counterfactual_metrics = (
    counterfactuals.groupby("strategy", as_index=False)
    .agg(
        explanations=("vehicle_id", "size"),
        feasible_share=("counterfactual_feasible", "mean"),
        median_changed_intervals=("changed_intervals", "median"),
        median_schedule_distance_kwh=("schedule_distance_kwh", "median"),
        mean_additional_surplus_window_energy_kwh=("additional_surplus_window_energy_kwh", "mean"),
    )
)
display(counterfactual_metrics)

,strategy,explanations,feasible_share,median_changed_intervals,median_schedule_distance_kwh,mean_additional_surplus_window_energy_kwh
0,optimised_actual,424,0.995283,64.0,33.228887,25.364479


## 4. Save outputs

In [7]:
save_table(counterfactuals, RESULTS_DIR / "counterfactual_recommendations.csv")
save_table(counterfactual_metrics, RESULTS_DIR / "counterfactual_metrics.csv")

Saved: Data/Results/counterfactual_recommendations.csv
Saved: Data/Results/counterfactual_metrics.csv


PosixPath('/Users/mac/Send-Ev-Project/SEND-EV-Project/Data/Results/counterfactual_metrics.csv')

## Interpretation

The recommendation is about the charging schedule, not the weather. Only use recommendations that pass the feasibility checks.